### Creating a view to analyse the rankings of the constructors over the season based on various metrics
- Sources -> fact_session_results, dim_drivers
- Required columns -> season, constructor_id, constructor_name, nationality
- Calculated columns -> race_starts (number of races the driver participates), total_points (sum of the points scored by the constructors), number_of_wins (if the is_win column is true), number_of_podiums (if the is_podium column is true)

In [0]:
%sql
CREATE OR REPLACE VIEW formula1.gold.v_constructor_standings AS
WITH constructor_session_summary AS 
(select 
f.season, d.constructor_id, d.constructor_name, d.nationality, 
COUNT(*) as race_starts,
SUM(f.points) as total_points,
count_if(f.is_win) as number_of_wins,
count_if(f.is_podium) as number_of_podiums from 
formula1.gold.fact_session_results f join formula1.gold.dim_constructors d on f.constructor_id= d.constructor_id
group by f.season, d.constructor_id, d.constructor_name, d.nationality)
select season, constructor_id, constructor_name, nationality,rank() over (partition by season order by total_points desc, number_of_wins desc) as standings, race_starts, total_points, number_of_wins, number_of_podiums from constructor_session_summary

In [0]:
%sql
select * from formula1.gold.v_constructor_standings order by season, standings